# NIFTY Options Backtest — 2024 Real Data

**Config: D_Bear10_Bull0** | Entry 09:25 | Exit 11:15 | Top 10 bearish signals

- Entry = real option OPEN at 09:25 candle
- TP = option HIGH hits target; SL = option LOW hits stop
- No Black-Scholes, no interpolation

**How to use:**
1. Edit Cell 1 (change SL, TP, lots)
2. Run Cell 2 once (builds cache — takes ~30s first time, instant after)
3. Run Cell 3 to see every trade
4. Run Cell 4 for summary table

In [43]:
# ── EDIT THESE ───────────────────────────────────────────────────────────────
SL_PCT     = 0.15   # stop loss   e.g. 0.15 = 15%
TP_PCT     = 0.40   # target      e.g. 0.40 = 40%
FIXED_LOTS = 2     # base lots per trade
MAX_LOTS   = 10   # cap as capital grows
DTE0_LOTS  = 5    # cap on expiry day (DTE=0)
# ─────────────────────────────────────────────────────────────────────────────

print(f"SL={SL_PCT:.0%}  TP={TP_PCT:.0%}  Fixed={FIXED_LOTS}L  Max={MAX_LOTS}L  DTE0={DTE0_LOTS}L")
print(f"Breakeven win rate: {SL_PCT/(SL_PCT+TP_PCT):.1%}")

SL=15%  TP=40%  Fixed=2L  Max=10L  DTE0=5L
Breakeven win rate: 27.3%


In [44]:
import sys, warnings, pickle
from pathlib import Path
from datetime import date, timedelta
from datetime import time as dtime
import pandas as pd
import numpy as np
import yfinance as yf
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────────
import os
cwd = Path(os.getcwd())
HERE = cwd / 'backtesting_2024_options' if cwd.name == 'market-research' else cwd
DATA_DIR    = HERE / '2024'
NIFTY_DIR   = DATA_DIR / '2024Nifty'
EXPIRY_CSV  = DATA_DIR / 'expiry.csv'
SIGNALS_CSV = HERE.parent / 'v2' / 'v2_reliable_signals.csv'
CACHE_FILE  = HERE / 'backtest_outputs' / 'trade_cache_D.pkl'
(HERE / 'backtest_outputs').mkdir(exist_ok=True)

# ── Constants (do NOT change) ─────────────────────────────────────────────────
LOT_SIZE    = 75
BROKERAGE   = 80        # Rs per trade
STRIKE_STEP = 50
ENTRY_TIME  = '09:25'
EXIT_TIME   = '11:15'
BEAR_N      = 10
BASE_RATE   = 54.5
GAP_TH      = 0.0015
GAP_LG_TH   = 0.0050
VIX_RIS_TH  = 0.03
VIX_SPK_TH  = 0.05
MONTH_ABBR  = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']

NSE_HOLIDAYS = {
    date(2024, 1,22), date(2024, 3,25), date(2024, 3,29),
    date(2024, 4,11), date(2024, 4,14), date(2024, 4,17),
    date(2024, 5, 1), date(2024, 6,17), date(2024, 8,15),
    date(2024,10, 2), date(2024,10,24), date(2024,11,15),
    date(2024,12,25),
}
EVENT_DAYS = {
    date(2024, 2, 1), date(2024, 4, 5), date(2024, 6, 4),
    date(2024, 6, 5), date(2024, 6, 7), date(2024, 7,23),
    date(2024, 8, 8), date(2024,10, 9), date(2024,12, 6),
}

def d2dmy(d):
    return f"{d.day:02d}{MONTH_ABBR[d.month-1]}{str(d.year)[2:]}"

def parse_dmy(s):
    return date(int(s[5:])+2000, MONTH_ABBR.index(s[2:5].upper())+1, int(s[:2]))

def is_trading(d):
    return d.weekday() < 5 and d not in NSE_HOLIDAYS

# ─────────────────────────────────────────────────────────────────────────────
# Load or build entry cache
# ─────────────────────────────────────────────────────────────────────────────
if CACHE_FILE.exists():
    print("Loading cached trade data...")
    with open(CACHE_FILE, 'rb') as f:
        entry_cache = pickle.load(f)
    # Rebuild candles DataFrames (stored as list of dicts for pickle compatibility)
    import pandas as _pd
    for _d, _v in entry_cache.items():
        if _v and isinstance(_v['candles'], list):
            _v['candles'] = _pd.DataFrame(_v['candles'])
    valid = sum(1 for v in entry_cache.values() if v is not None)
    print(f"  {valid} tradeable days loaded from cache.")
    print("  (Delete backtest_outputs/trade_cache_D.pkl to rebuild from scratch)")
else:
    print("Building trade cache — first run takes ~30 seconds...")

    # 1. NIFTY spot
    chunks = []
    for fp in sorted(NIFTY_DIR.glob('Nifty-2024*.csv')):
        df = pd.read_csv(fp, header=0,
                         names=['datetime','open','high','low','close','volume'],
                         skiprows=1)
        df['datetime'] = pd.to_datetime(df['datetime'], format='%Y-%m-%d %H:%M', errors='coerce')
        df.dropna(subset=['datetime'], inplace=True)
        chunks.append(df)
    nifty_spot = pd.concat(chunks).sort_values('datetime').reset_index(drop=True)
    nifty_spot['date'] = nifty_spot['datetime'].dt.date
    nifty_spot['time'] = nifty_spot['datetime'].dt.time

    daily = (nifty_spot[nifty_spot['time']==dtime(9,15)][['date','open']]
             .rename(columns={'open':'nifty_open'})
             .merge(nifty_spot.groupby('date')['close'].last().reset_index()
                              .rename(columns={'close':'nifty_close'}), on='date')
             .sort_values('date').reset_index(drop=True))
    print(f"  NIFTY spot: {len(daily)} days")

    # 2. Expiry schedule
    raw_exp = pd.read_csv(EXPIRY_CSV, header=0)
    expiry_dates = sorted([
        parse_dmy(s.strip()) for s in raw_exp.iloc[:,0].dropna().astype(str)
        if len(s.strip())==7 and s.strip()[:2].isdigit() and parse_dmy(s.strip()).year==2024
    ])

    def nearest_expiry(d):
        for e in expiry_dates:
            if e >= d: return e
        return None

    # 3. Global signals
    print("  Fetching global data from yfinance...")
    TICKERS = {'SP500':'^GSPC','SGX':'NKD=F','DAX':'^GDAXI','VIX_US':'^VIX','VIX_INDIA':'^INDIAVIX'}
    gdf = None
    for name, ticker in TICKERS.items():
        try:
            df = yf.download(ticker, start='2023-12-01', end='2025-01-05',
                             progress=False, auto_adjust=True)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            df = df[['Close']].rename(columns={'Close': name})
            df.index = pd.to_datetime(df.index).normalize()
            gdf = df if gdf is None else gdf.join(df, how='outer')
        except: pass
    gdf = gdf.sort_index().ffill()

    def _ret(col, ts):
        s = gdf[gdf.index < ts][col].dropna() if col in gdf.columns else pd.Series()
        return float((s.iloc[-1]-s.iloc[-2])/s.iloc[-2]) if len(s)>=2 else 0.0

    # 4. Build daily signals dataframe
    sig_rows = []
    ds = daily.sort_values('date').reset_index(drop=True)
    for i, row in ds.iterrows():
        d = row['date']
        if not isinstance(d, date): d = d.date()
        if d.year != 2024 or not is_trading(d): continue
        prev = ds[ds['date'] < d]
        if len(prev) < 2: continue
        nifty_open  = float(row['nifty_open'])
        prev_close  = float(prev.iloc[-1]['nifty_close'])
        pp_close    = float(prev.iloc[-2]['nifty_close'])
        gap         = (nifty_open - prev_close) / prev_close
        prev_india  = (prev_close - pp_close) / pp_close
        ts = pd.Timestamp(d)
        sig_rows.append({
            'date': d, 'nifty_open': nifty_open,
            'gap':      gap,
            'pind':     prev_india,
            'sp500':    _ret('SP500',    ts),
            'sgx':      _ret('SGX',      ts),
            'dax':      _ret('DAX',      ts),
            'vix':      _ret('VIX_US',   ts),
        })
    sig_df = pd.DataFrame(sig_rows)
    print(f"  Signal days: {len(sig_df)}")

    # 5. Load signal combos
    rel = pd.read_csv(SIGNALS_CSV)
    bear_combos = list(
        rel[rel['P_Down'] > BASE_RATE].sort_values('Edge_pp', ascending=False)['Signal'][:BEAR_N]
    )

    def sig_map(r):
        return {
            'Gap Up':          r['gap'] >  GAP_TH,
            'Gap Up Strong':   r['gap'] >  GAP_LG_TH,
            'Gap Down':        r['gap'] < -GAP_TH,
            'Prev India UP':   r['pind'] > 0,
            'Prev India DOWN': r['pind'] < 0,
            'US UP':           r['sp500'] > 0,
            'US DOWN':         r['sp500'] < 0,
            'SGX UP':          r['sgx']  > 0,
            'SGX DOWN':        r['sgx']  < 0,
            'DAX UP':          r['dax']  > 0,
            'VIX Rising':      r['vix']  > VIX_RIS_TH,
            'VIX Falling':     r['vix']  < 0,
            'VIX Spike':       r['vix']  > VIX_SPK_TH,
        }

    def fires(sigs, combo):
        return all(sigs.get(s.strip(), False) for s in combo.split('+'))

    # 6. Build entry cache
    _opt_cache = {}
    def load_opt(d, expiry):
        key = (d, expiry)
        if key in _opt_cache: return _opt_cache[key]
        mon = MONTH_ABBR[d.month-1]
        fp  = DATA_DIR / f'2024{mon}' / f'NIFTY-{d2dmy(expiry)}-{d2dmy(d)}.csv'
        if not fp.exists(): _opt_cache[key]=None; return None
        df = pd.read_csv(fp); df.columns=[c.strip() for c in df.columns]
        _opt_cache[key] = df; return df

    entry_cache = {}
    for _, row in sig_df.iterrows():
        d = row['date']
        if d.weekday()==0 or d in EVENT_DAYS: continue
        sigs = sig_map(row)
        bf = [c for c in bear_combos if fires(sigs, c)]
        if not bf: continue

        expiry = nearest_expiry(d)
        if not expiry: continue
        dte  = (expiry - d).days
        atm  = round(row['nifty_open']/STRIKE_STEP)*STRIKE_STEP
        strike = atm - STRIKE_STEP

        df = load_opt(d, expiry)
        if df is None: entry_cache[d]=None; continue

        mask = (df['strike_price']==strike) & (df['right']=='PE')
        at_entry = df[mask & (df['datetime']==ENTRY_TIME)]
        if at_entry.empty:
            at_entry = df[mask & (df['datetime']>=ENTRY_TIME)].sort_values('datetime').head(1)
        if at_entry.empty: entry_cache[d]=None; continue
        ep = float(at_entry.iloc[0]['open'])
        if ep <= 0: entry_cache[d]=None; continue

        cands = df[mask].copy()
        cands = cands[(cands['datetime']>=ENTRY_TIME) & (cands['datetime']<=EXIT_TIME)]
        cands = cands.sort_values('datetime').reset_index(drop=True)
        if cands.empty: entry_cache[d]=None; continue

        entry_cache[d] = {
            'expiry': expiry, 'dte': dte, 'strike': int(strike),
            'entry_price': ep, 'candles': cands.to_dict('records'),  # plain list — pickle-safe
            'nifty_open': row['nifty_open'], 'combo': bf[0],
        }

    valid = sum(1 for v in entry_cache.values() if v is not None)
    print(f"  Entry cache: {valid} valid days")

    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(entry_cache, f)
    print("  Saved to cache.")

valid_days = sorted(d for d, v in entry_cache.items() if v is not None)
print(f"\nReady: {len(valid_days)} tradeable days  ({valid_days[0]} to {valid_days[-1]})")

Loading cached trade data...
  60 tradeable days loaded from cache.
  (Delete backtest_outputs/trade_cache_D.pkl to rebuild from scratch)

Ready: 60 tradeable days  (2024-01-04 to 2024-12-05)


In [45]:
# ── Run simulation ────────────────────────────────────────────────────────────
W = 64   # print width

first_info  = entry_cache[valid_days[0]]
first_lots  = min(FIXED_LOTS, DTE0_LOTS) if first_info['dte']==0 else FIXED_LOTS
init_cap    = first_info['entry_price'] * LOT_SIZE * first_lots
init_cap    = max(init_cap, first_info['entry_price'] * LOT_SIZE)

capital     = init_cap
peak        = capital
max_dd      = 0.0
refill_count= 0
total_refill= 0.0
wins = losses = time_exits = 0
trade_log   = []

print(f"Initial capital : Rs {init_cap:,.0f}  ({first_lots} lots x {first_info['entry_price']:.2f} x {LOT_SIZE})")
print(f"SL={SL_PCT:.0%}  TP={TP_PCT:.0%}  Fixed={FIXED_LOTS}L  Max={MAX_LOTS}L  DTE0={DTE0_LOTS}L")
print("=" * W)

for n, d in enumerate(valid_days, 1):
    info        = entry_cache[d]
    ep          = info['entry_price']
    cost_1lot   = ep * LOT_SIZE

    # Refill if can't afford FIXED_LOTS (minimum trade size)
    min_lots  = DTE0_LOTS if info["dte"]==0 else FIXED_LOTS
    min_cost  = cost_1lot * min_lots
    if capital < min_cost:
        refill        = min_cost - capital   # add just enough for min_lots
        capital      += refill
        peak          = max(peak, capital)
        refill_count += 1
        total_refill += refill
        print(f"      *** REFILL ***  Capital Rs {capital-refill:,.0f} < Rs {min_cost:,.0f} ({min_lots} lots min)")
        print(f"{'':>4}  Added Rs {refill:,.0f}  -> Rs {capital:,.0f}")
        print("-" * W)

    # Lot sizing
    affordable = max(int(capital // cost_1lot), 1)  # lots = floor(capital / cost_per_lot)
    lots       = min(affordable, MAX_LOTS)
    if info['dte'] == 0:
        lots = min(lots, DTE0_LOTS)
    lots = max(lots, 1)

    sl_price = ep * (1 - SL_PCT)
    tp_price = ep * (1 + TP_PCT)

    # Simulate candle by candle
    exit_price  = None
    exit_reason = f'{EXIT_TIME} exit'
    exit_t      = EXIT_TIME
    for _, row in info['candles'].iterrows():
        t   = str(row['datetime'])
        lo  = float(row['low'])
        hi  = float(row['high'])
        if hi >= tp_price:
            exit_price, exit_reason, exit_t = tp_price, 'TARGET HIT', t; break
        if lo <= sl_price:
            exit_price, exit_reason, exit_t = sl_price, 'STOP LOSS',  t; break
    if exit_price is None:
        row_ex = info['candles'][info['candles']['datetime']==EXIT_TIME]
        if not row_ex.empty:
            exit_price = float(row_ex.iloc[0]['close'])
        else:
            last = info['candles'][info['candles']['datetime']<=EXIT_TIME]
            exit_price = float(last.iloc[-1]['close']) if not last.empty else ep

    pnl_pts = round(exit_price - ep, 2)
    pnl_rs  = round(pnl_pts * lots * LOT_SIZE - BROKERAGE, 0)
    capital_before = capital
    capital += pnl_rs

    # Drawdown
    if capital > peak: peak = capital
    dd = (peak - capital) / peak * 100
    if dd > max_dd: max_dd = dd

    # Counters
    if exit_reason == 'TARGET HIT': wins      += 1
    elif exit_reason == 'STOP LOSS': losses   += 1
    else:                            time_exits+= 1

    # Print
    result_tag = 'WIN ' if pnl_rs > 0 else 'LOSS'
    print(f"Trade {n:>3}  {d}  |  {info['strike']} PE  DTE={info['dte']}  |  Entry={ep:.2f}")
    print(f"        Capital Rs {capital_before:>10,.0f}  ->  {lots} lots  |  SL={sl_price:.2f}  TP={tp_price:.2f}")
    print(f"  {result_tag}  {exit_reason:<12} @ {exit_t}  ->  exit {exit_price:.2f}")
    pnl_detail = f"{pnl_pts:+.2f} x {lots}L x {LOT_SIZE} - {BROKERAGE} brok"
    print(f"        P&L : {pnl_detail} = Rs {pnl_rs:>+,.0f}")
    print(f"        Capital after : Rs {capital:,.0f}   DD={dd:.1f}%")
    print("-" * W)

    trade_log.append({
        'Trade': n, 'Date': d, 'Strike': info['strike'], 'DTE': info['dte'],
        'Entry': ep, 'Lots': lots, 'SL': round(sl_price,2), 'TP': round(tp_price,2),
        'Exit': exit_price, 'Exit Reason': exit_reason, 'Exit Time': exit_t,
        'PnL pts': pnl_pts, 'PnL Rs': pnl_rs,
        'Capital After': round(capital, 0),
    })

print("=" * W)
print(f"Done.  {len(valid_days)} trades  |  Wins={wins}  Losses={losses}  Time exits={time_exits}")

Initial capital : Rs 2,992  (2 lots x 19.95 x 75)
SL=15%  TP=40%  Fixed=2L  Max=10L  DTE0=5L
      *** REFILL ***  Capital Rs 2,992 < Rs 7,481 (5 lots min)
      Added Rs 4,489  -> Rs 7,481
----------------------------------------------------------------
Trade   1  2024-01-04  |  21550 PE  DTE=0  |  Entry=19.95
        Capital Rs      7,481  ->  5 lots  |  SL=16.96  TP=27.93
  LOSS  STOP LOSS    @ 09:28  ->  exit 16.96
        P&L : -2.99 x 5L x 75 - 80 brok = Rs -1,201
        Capital after : Rs 6,280   DD=16.1%
----------------------------------------------------------------
      *** REFILL ***  Capital Rs 6,280 < Rs 12,315 (2 lots min)
      Added Rs 6,035  -> Rs 12,315
----------------------------------------------------------------
Trade   2  2024-01-05  |  21650 PE  DTE=6  |  Entry=82.10
        Capital Rs     12,315  ->  2 lots  |  SL=69.78  TP=114.94
  WIN   11:15 exit   @ 11:15  ->  exit 94.30
        P&L : +12.20 x 2L x 75 - 80 brok = Rs +1,750
        Capital after : Rs 14,

In [46]:
# ── Summary ───────────────────────────────────────────────────────────────────
from datetime import date as _date
import numpy as np

n_trades  = wins + losses + time_exits
win_pct   = wins / n_trades * 100 if n_trades else 0
total_pnl = sum(t['PnL Rs'] for t in trade_log)
avg_pnl   = total_pnl / n_trades if n_trades else 0
breakeven = SL_PCT / (SL_PCT + TP_PCT) * 100

# XIRR
t0 = valid_days[0]
cash_flows = [(-init_cap, t0)]
if total_refill > 0:
    cash_flows.append((-total_refill, valid_days[-1]))
cash_flows.append((capital, _date(2024, 12, 31)))

def npv(r):
    return sum(amt / (1+r)**((d-t0).days/365.25) for amt, d in cash_flows)

xirr = None
try:
    lo, hi = -0.999, 100.0
    for _ in range(300):
        mid = (lo+hi)/2
        if npv(mid) > 0: lo = mid
        else:            hi = mid
        if hi-lo < 1e-7: break
    xirr = (lo+hi)/2 * 100
except: pass

print("=" * 50)
print("  BACKTEST SUMMARY — 2024 Real Data")
print("=" * 50)
print(f"  Config       : SL={SL_PCT:.0%}  TP={TP_PCT:.0%}  {FIXED_LOTS}L fixed / {MAX_LOTS}L max / {DTE0_LOTS}L DTE0")
print(f"  Breakeven    : {breakeven:.1f}%  (your win rate: {win_pct:.1f}%)")
print()
print(f"  Trades       : {n_trades}")
print(f"  Wins         : {wins}  ({win_pct:.1f}%)")
print(f"  Losses       : {losses}")
print(f"  Time exits   : {time_exits}")
print()
print(f"  Total PnL    : Rs {total_pnl:>+,.0f}")
print(f"  Avg per trade: Rs {avg_pnl:>+,.0f}")
print()
print(f"  Start capital: Rs {init_cap:,.0f}")
print(f"  End capital  : Rs {capital:,.0f}")
print(f"  Net return   : {(capital-init_cap-total_refill)/init_cap*100:+.1f}%")
print(f"  XIRR         : {xirr:+.1f}%" if xirr else "  XIRR         : N/A")
print(f"  Max drawdown : {max_dd:.1f}%")
print()
print(f"  Refills      : {refill_count}  (Rs {total_refill:,.0f} added)")
print("=" * 50)

# Monthly breakdown
df_log = pd.DataFrame(trade_log)
df_log['Month'] = pd.to_datetime(df_log['Date']).dt.month
mo = df_log.groupby('Month').agg(
    Trades=('PnL Rs','count'),
    Wins=('PnL Rs', lambda x: (x>0).sum()),
    PnL_Rs=('PnL Rs','sum')
).reset_index()
mo['Win%'] = (mo['Wins']/mo['Trades']*100).round(1)
mo['Month'] = mo['Month'].map({1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
                                7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'})
print()
print(mo.to_string(index=False))

  BACKTEST SUMMARY — 2024 Real Data
  Config       : SL=15%  TP=40%  2L fixed / 10L max / 5L DTE0
  Breakeven    : 27.3%  (your win rate: 40.0%)

  Trades       : 60
  Wins         : 24  (40.0%)
  Losses       : 33
  Time exits   : 3

  Total PnL    : Rs +143,445
  Avg per trade: Rs +2,391

  Start capital: Rs 2,992
  End capital  : Rs 157,292
  Net return   : +4793.5%
  XIRR         : +4846.1%
  Max drawdown : 40.9%

  Refills      : 3  (Rs 10,854 added)

Month  Trades  Wins   PnL_Rs  Win%
  Jan       7     3   3052.0  42.9
  Feb       8     5  22823.0  62.5
  Mar       4     2  23350.0  50.0
  Apr       6     2 -14868.0  33.3
  May       3     2  17880.0  66.7
  Jun       5     1 -13822.0  20.0
  Jul       4     1   2716.0  25.0
  Aug       7     3  32093.0  42.9
  Sep       5     1 -18612.0  20.0
  Oct       3     2  27892.0  66.7
  Nov       5     3  76901.0  60.0
  Dec       3     1 -15960.0  33.3
